# SmolVLM-500M Two-Stage Fine-Tuning on ScienceQA

Inspired by **KAM-CoT** (Mondal et al., 2024), this notebook fine-tunes
`HuggingFaceTB/SmolVLM-500M-Instruct` using a two-stage chain-of-thought approach:

| Stage | Input | Target |
|---|---|---|
| **Stage 1** | image + lecture + hint + question + choices | `solution` (rationale) |
| **Stage 2** | image + lecture + hint + question + choices + **rationale** | answer letter (A/B/C/D/E) |

**Expected folder layout:**
```
data/
  train.csv
  val.csv
  test.csv
  sample_submission.csv
  images/
    train/   <- train images
    val/     <- val images
    test/    <- test images
```

**Requirements:** GPU with ≥16 GB VRAM. Set `USE_4BIT = True` for smaller GPUs.

## 1. Install Dependencies

In [ ]:
!pip install -q \
    transformers>=4.45.0 \
    trl>=0.12.0 \
    peft>=0.13.0 \
    datasets \
    accelerate \
    bitsandbytes \
    pillow \
    tqdm

## 2. Imports & Config

In [27]:
import os
import ast
import json
import random
import gc
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
from datasets import Dataset
from transformers import (
    AutoProcessor,
    AutoModelForVision2Seq,
    BitsAndBytesConfig,
    EvalPrediction,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from transformers import Trainer, TrainingArguments

# ── Reproducibility ───────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Paths ─────────────────────────────────────────────────────
MODEL_ID    = 'HuggingFaceTB/SmolVLM-500M-Instruct'
TRAIN_CSV   = 'data/train.csv'
VAL_CSV     = 'data/val.csv'
TEST_CSV    = 'data/test.csv'
IMAGE_BASE  = 'data'   # folder that contains images/train/, images/val/, images/test/
STAGE1_DIR  = 'smolvlm-stage1-rationale'
STAGE2_DIR  = 'smolvlm-stage2-answer'

# ── Task ──────────────────────────────────────────────────────
LETTERS = 'ABCDE'

# ── Training hyper-params (shared across both stages) ─────────
NUM_EPOCHS        = 1    # KAM-CoT trained for 20 epochs; use 3-5 for quick tests
PER_DEVICE_BATCH  = 1     # KAM-CoT used 1; increase if VRAM allows
GRAD_ACCUM        = 8     # effective batch = 8
LEARNING_RATE     = 5e-5  # matches KAM-CoT
WARMUP_RATIO      = 0.05
LR_SCHEDULER      = 'cosine'
MAX_SEQ_LEN       = 512   # KAM-CoT used 512 input tokens
MAX_NEW_RATIONALE = 512   # generation budget for Stage 1 (rationale)
MAX_NEW_ANSWER    = 10    # generation budget for Stage 2 (answer letter)

# ── LoRA ──────────────────────────────────────────────────────
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

# ── Hardware ──────────────────────────────────────────────────
USE_4BIT = False  # set True for GPUs with <16 GB VRAM
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16

print(f'Device : {DEVICE}')
print(f'BF16   : {USE_BF16}  |  FP16: {USE_FP16}  |  4-bit: {USE_4BIT}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device : cuda
BF16   : True  |  FP16: False  |  4-bit: False
GPU    : NVIDIA GeForce RTX 3090
VRAM   : 25.8 GB


## 3. Load Data

In [2]:
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

print(f'Train : {len(train_df)} rows')
print(f'Val   : {len(val_df)} rows')
print(f'Test  : {len(test_df)} rows')
print(f'Columns: {train_df.columns.tolist()}')
print()
print('Train null counts:')
print(train_df.isnull().sum())
print()
print('Val null counts:')
print(val_df.isnull().sum())

Train : 3109 rows
Val   : 1048 rows
Test  : 1008 rows
Columns: ['id', 'image_path', 'question', 'choices', 'num_choices', 'answer', 'hint', 'lecture', 'solution', 'task', 'grade', 'subject', 'topic', 'category', 'skill']

Train null counts:
id               0
image_path       0
question         0
choices          0
num_choices      0
answer           0
hint           724
lecture        440
solution       529
task             0
grade            0
subject          0
topic            0
category         0
skill            0
dtype: int64

Val null counts:
id               0
image_path       0
question         0
choices          0
num_choices      0
answer           0
hint           232
lecture        133
solution       172
task             0
grade            0
subject          0
topic            0
category         0
skill            0
dtype: int64


## 4. Build Training Records

- **Stage 1:** target is the `solution` column (chain-of-thought rationale). Rows with no solution are skipped.
- **Stage 2:** target is the answer letter. The gold `solution` is appended to the prompt as `Reasoning:` context.

In [3]:
def build_base_parts(row):
    """Shared prompt prefix used by both stages."""
    choices = ast.literal_eval(row['choices'])
    choices_str = '\n'.join(f'{LETTERS[i]}) {c}' for i, c in enumerate(choices))
    lecture = str(row['lecture']).strip() if pd.notna(row['lecture']) else ''
    hint    = str(row['hint']).strip()    if pd.notna(row['hint'])    else ''
    parts = []
    if lecture:
        parts.append(f'Lecture:\n{lecture}')
    if hint:
        parts.append(f'Hint:\n{hint}')
    parts.append(f'Question:\n{row["question"]}\n\nChoices:\n{choices_str}')
    return parts


def row_to_stage1(row):
    """Stage 1 record. Returns None if the row has no solution."""
    solution = str(row['solution']).strip() if pd.notna(row['solution']) else ''
    if not solution:
        return None
    parts = build_base_parts(row)
    parts.append('Explain your reasoning step by step before stating the answer.')
    return {
        'image_path': os.path.join(IMAGE_BASE, row['image_path']),
        'user_text' : '\n\n'.join(parts),
        'answer'    : solution,
    }


def row_to_stage2(row):
    """Stage 2 record. Uses gold solution in prompt; target is the answer letter."""
    solution = str(row['solution']).strip() if pd.notna(row['solution']) else ''
    parts = build_base_parts(row)
    if solution:
        parts.append(f'Reasoning:\n{solution}')
    parts.append('Answer with the single letter of the correct choice (e.g. A).')
    return {
        'image_path': os.path.join(IMAGE_BASE, row['image_path']),
        'user_text' : '\n\n'.join(parts),
        'answer'    : LETTERS[int(row['answer'])],
    }


# Build from train
s1_train = [r for _, row in train_df.iterrows() if (r := row_to_stage1(row)) is not None]
s2_train = [row_to_stage2(row) for _, row in train_df.iterrows()]

# Build from val
s1_val = [r for _, row in val_df.iterrows() if (r := row_to_stage1(row)) is not None]
s2_val = [row_to_stage2(row) for _, row in val_df.iterrows()]

print(f'Stage 1 — train: {len(s1_train)}  |  val: {len(s1_val)}')
print(f'Stage 2 — train: {len(s2_train)}  |  val: {len(s2_val)}')
print()
print('── Stage 1 sample prompt (first 500 chars) ──')
print(s1_train[0]['user_text'][:500])
print('\n── Stage 1 sample target (first 200 chars) ──')
print(s1_train[0]['answer'][:200])
print('\n── Stage 2 sample target ──', s2_train[0]['answer'])

Stage 1 — train: 2580  |  val: 876
Stage 2 — train: 3109  |  val: 1048

── Stage 1 sample prompt (first 500 chars) ──
Lecture:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survi

── Stage 1 sample target (first 200 chars) ──
Look for the part of the passage that describes the effect of putting each tadpole in its own pool of water. Use this information to determine why this behavior can increase the reproductive success o

── Stage 2 sample target ── C


## 5. Wrap as HuggingFace Datasets

In [4]:
s1_train_ds = Dataset.from_list(s1_train)
s1_val_ds   = Dataset.from_list(s1_val)
s2_train_ds = Dataset.from_list(s2_train)
s2_val_ds   = Dataset.from_list(s2_val)

print(f'Stage 1 — train: {len(s1_train_ds)}  |  val: {len(s1_val_ds)}')
print(f'Stage 2 — train: {len(s2_train_ds)}  |  val: {len(s2_val_ds)}')

Stage 1 — train: 2580  |  val: 876
Stage 2 — train: 3109  |  val: 1048


## 6. Shared Utilities

In [21]:
# ── Processor ─────────────────────────────────────────────────
processor = AutoProcessor.from_pretrained(MODEL_ID)
print('Processor:', type(processor).__name__)


# ── Model loader ──────────────────────────────────────────────
def load_base_model():
    """Load a fresh copy of SmolVLM from the Hub."""
    if USE_4BIT:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        return AutoModelForVision2Seq.from_pretrained(
            MODEL_ID, quantization_config=bnb_config,
            device_map='auto', trust_remote_code=True,
        )
    dtype = torch.bfloat16 if USE_BF16 else torch.float16 if USE_FP16 else torch.float32
    return AutoModelForVision2Seq.from_pretrained(
        MODEL_ID, torch_dtype=dtype,
        device_map='auto', trust_remote_code=True,
    )


# ── LoRA wrapper ──────────────────────────────────────────────
def apply_lora(model):
    lora_cfg = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        bias='none', target_modules=['model.text_model.layers.' + str(i) + '.self_attn.' + proj
                for i in range(32) for proj in ['q_proj', 'k_proj', 'v_proj', 'o_proj']], task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    return model


# ── Data collator ─────────────────────────────────────────────
def make_collator(max_len):
    def collate_fn(examples):
        texts, images = [], []
        for ex in examples:
            try:
                img = Image.open(ex['image_path']).convert('RGB')
            except FileNotFoundError:
                img = Image.new('RGB', (224, 224), (128, 128, 128))
                print(f"[WARN] Missing image: {ex['image_path']}")
            messages = [
                {'role': 'user', 'content': [
                    {'type': 'image'},
                    {'type': 'text', 'text': ex['user_text']},
                ]},
                {'role': 'assistant', 'content': [
                    {'type': 'text', 'text': ex['answer']},
                ]},
            ]
            texts.append(processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            ))
            images.append([img])
        batch = processor(
            text=texts, images=images,
            return_tensors='pt', padding=True,
        )
        labels = batch['input_ids'].clone()
        labels[labels == processor.tokenizer.pad_token_id] = -100
        batch['labels'] = labels
        return batch
    return collate_fn


# ── Accuracy metric ───────────────────────────────────────────
from transformers import TrainerCallback

class ValAccuracyCallback(TrainerCallback):
    def __init__(self, val_records, processor, model, n_samples=100):
        # sample a fixed subset so it's fast
        self.samples = random.sample(val_records, min(n_samples, len(val_records)))
        self.processor = processor
        self.model = model

    def on_epoch_end(self, args, state, control, **kwargs):
        self.model.eval()
        correct = 0
        for ex in self.samples:
            img = Image.open(ex['image_path']).convert('RGB')
            messages = [{'role': 'user', 'content': [
                {'type': 'image'},
                {'type': 'text', 'text': ex['user_text']},
            ]}]
            prompt = self.processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            inputs = self.processor(
                text=[prompt], images=[[img]], return_tensors='pt'
            ).to(self.model.device)
            with torch.no_grad():
                out = self.model.generate(**inputs, max_new_tokens=MAX_NEW_ANSWER, do_sample=False)
            decoded = self.processor.decode(
                out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
            ).strip()
            pred = decoded[0].upper() if decoded and decoded[0].upper() in LETTERS else '?'
            correct += int(pred == ex['answer'])

        acc = correct / len(self.samples)
        print(f'\n[Epoch {state.epoch:.0f}] Val accuracy (n={len(self.samples)}): {acc:.2%}')
        self.model.train()


# ── Training args factory ─────────────────────────────────────
def make_training_args(output_dir):
    return TrainingArguments(
        output_dir=output_dir,
        overwrite_output_dir=True,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_BATCH,
        per_device_eval_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type=LR_SCHEDULER,
        weight_decay=0.01,
        bf16=USE_BF16,
        fp16=USE_FP16,
        logging_steps=10,
        eval_strategy='no',
        save_strategy='epoch',
        load_best_model_at_end=False,
        metric_for_best_model='accuracy',
        greater_is_better=True,
        remove_unused_columns=False,  # CRITICAL for VLMs
        dataloader_num_workers=0,
        dataloader_pin_memory=True,
        report_to='none',
        seed=SEED,
        max_grad_norm=1.0,  # already likely there
    )


print('All utilities defined.')

Processor: Idefics3Processor
All utilities defined.


---
## 7. Stage 1 — Train Rationale Generator

In [22]:
print('=' * 60)
print('STAGE 1: Rationale Generation')
print('=' * 60)

s1_model   = load_base_model()
s1_model   = apply_lora(s1_model)
s1_trainer = Trainer(
    model=s1_model,
    args=make_training_args(STAGE1_DIR),
    train_dataset=s1_train_ds,
    data_collator=make_collator(MAX_SEQ_LEN),
    callbacks=[ValAccuracyCallback(s1_val, processor, s1_model, n_samples=100)],
)

print(f'Training on {len(s1_train_ds)} examples for {NUM_EPOCHS} epochs...')
s1_result = s1_trainer.train()
print(f'Train loss : {s1_result.metrics["train_loss"]:.4f}')
print(f'Runtime    : {s1_result.metrics["train_runtime"]:.0f}s')

STAGE 1: Rationale Generation


The model is already on multiple devices. Skipping the move to device specified in `args`.


trainable params: 3,276,800 || all params: 510,759,104 || trainable%: 0.6416
Training on 2580 examples for 20 epochs...


Step,Training Loss
10,14.568900
20,14.666000
30,14.454000
40,14.514600
50,14.664900
60,14.674500
70,14.624100
80,14.369900
90,13.685000
100,13.050300



[Epoch 1] Val accuracy (n=100): 0.00%


KeyboardInterrupt: 

In [23]:
s1_trainer.save_model(STAGE1_DIR)
processor.save_pretrained(STAGE1_DIR)

['smolvlm-stage1-rationale/processor_config.json']

In [24]:
s1_eval = s1_trainer.evaluate()
print('Stage 1 val metrics:')
for k, v in s1_eval.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

ValueError: Trainer: evaluation requires an eval_dataset.

In [25]:
s1_trainer.save_model(STAGE1_DIR)
processor.save_pretrained(STAGE1_DIR)

all_s1_metrics = {**s1_result.metrics, **s1_eval}
with open(os.path.join(STAGE1_DIR, 'metrics.json'), 'w') as fh:
    json.dump(all_s1_metrics, fh, indent=2)

print(f'Stage 1 saved to {STAGE1_DIR}/')

# Free VRAM before Stage 2
del s1_trainer, s1_model
gc.collect()
torch.cuda.empty_cache()
print('VRAM cleared.')

NameError: name 's1_result' is not defined

---
## 8. Stage 2 — Train Answer Predictor

In [ ]:
print('=' * 60)
print('STAGE 2: Answer Prediction')
print('=' * 60)

# Fresh base model — independent from Stage 1
s2_model   = load_base_model()
s2_model   = apply_lora(s2_model)
s2_trainer = Trainer(
    model=s2_model,
    args=make_training_args(STAGE2_DIR),
    train_dataset=s2_train_ds,
    eval_dataset=s2_val_ds,
    data_collator=make_collator(MAX_SEQ_LEN),
    compute_metrics=compute_accuracy,
)

print(f'Training on {len(s2_train_ds)} examples for {NUM_EPOCHS} epochs...')
s2_result = s2_trainer.train()
print(f'Train loss : {s2_result.metrics["train_loss"]:.4f}')
print(f'Runtime    : {s2_result.metrics["train_runtime"]:.0f}s')

STAGE 2: Answer Prediction


The model is already on multiple devices. Skipping the move to device specified in `args`.


trainable params: 3,276,800 || all params: 510,759,104 || trainable%: 0.6416
Training on 3109 examples for 1 epochs...


Step,Training Loss
10,14.516400
20,14.519900
30,13.026500
40,11.627000
50,9.674100
60,7.854000
70,6.096800


In [ ]:
s2_eval = s2_trainer.evaluate()
print('Stage 2 val metrics:')
for k, v in s2_eval.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

In [ ]:
s2_trainer.save_model(STAGE2_DIR)
processor.save_pretrained(STAGE2_DIR)

all_s2_metrics = {**s2_result.metrics, **s2_eval}
with open(os.path.join(STAGE2_DIR, 'metrics.json'), 'w') as fh:
    json.dump(all_s2_metrics, fh, indent=2)

print(f'Stage 2 saved to {STAGE2_DIR}/')

---
## 9. Load Both Models for Inference

Run this section independently (after training) to reload the pipeline from disk.

In [ ]:
infer_dtype = torch.bfloat16 if USE_BF16 else torch.float32

base1 = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, torch_dtype=infer_dtype, device_map='auto', trust_remote_code=True
)
stage1_model = PeftModel.from_pretrained(base1, STAGE1_DIR).eval()

base2 = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, torch_dtype=infer_dtype, device_map='auto', trust_remote_code=True
)
stage2_model = PeftModel.from_pretrained(base2, STAGE2_DIR).eval()

infer_processor = AutoProcessor.from_pretrained(STAGE2_DIR)
print('Both models loaded.')

## 10. Two-Stage Inference Function

In [ ]:
def _generate(model, prompt_text, image, max_new_tokens):
    """Run one model forward pass; return decoded new tokens only."""
    messages = [{'role': 'user', 'content': [
        {'type': 'image'},
        {'type': 'text', 'text': prompt_text},
    ]}]
    prompt = infer_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = infer_processor(
        text=[prompt], images=[[image]], return_tensors='pt'
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    return infer_processor.decode(new_tokens, skip_special_tokens=True).strip()


def predict_two_stage(image_path, question, choices, lecture='', hint=''):
    """
    Run two-stage inference for one example.
    Returns (predicted_letter, rationale, raw_stage2_output).
    """
    img = Image.open(image_path).convert('RGB')
    choices_str = '\n'.join(f'{LETTERS[i]}) {c}' for i, c in enumerate(choices))

    # Stage 1 — generate rationale
    parts1 = []
    if lecture: parts1.append(f'Lecture:\n{lecture}')
    if hint:    parts1.append(f'Hint:\n{hint}')
    parts1.append(f'Question:\n{question}\n\nChoices:\n{choices_str}')
    parts1.append('Explain your reasoning step by step before stating the answer.')
    rationale = _generate(stage1_model, '\n\n'.join(parts1), img, MAX_NEW_RATIONALE)

    # Stage 2 — predict answer letter
    parts2 = []
    if lecture:   parts2.append(f'Lecture:\n{lecture}')
    if hint:      parts2.append(f'Hint:\n{hint}')
    parts2.append(f'Question:\n{question}\n\nChoices:\n{choices_str}')
    if rationale: parts2.append(f'Reasoning:\n{rationale}')
    parts2.append('Answer with the single letter of the correct choice (e.g. A).')
    raw = _generate(stage2_model, '\n\n'.join(parts2), img, MAX_NEW_ANSWER)

    pred = raw[0].upper() if raw and raw[0].upper() in LETTERS else '?'
    return pred, rationale, raw


print('predict_two_stage() ready.')

## 11. Mini Validation Eval (10 examples)

In [ ]:
n_eval   = 10
correct  = 0
sample   = val_df.sample(n=n_eval, random_state=SEED).reset_index(drop=True)

for idx, row in sample.iterrows():
    choices  = ast.literal_eval(row['choices'])
    lecture  = str(row['lecture']) if pd.notna(row['lecture']) else ''
    hint     = str(row['hint'])    if pd.notna(row['hint'])    else ''
    expected = LETTERS[int(row['answer'])]

    pred, rationale, _ = predict_two_stage(
        image_path=os.path.join(IMAGE_BASE, row['image_path']),
        question=row['question'],
        choices=choices,
        lecture=lecture,
        hint=hint,
    )
    ok = pred == expected
    correct += int(ok)
    print(f'[{idx+1:02d}] Expected: {expected}  Predicted: {pred}  {"✓" if ok else "✗"}')
    print(f'      Rationale (first 100 chars): {rationale[:100]}...')

print(f'\nMini-eval accuracy: {correct}/{n_eval} = {correct/n_eval:.0%}')

## 12. Full Validation Sweep (Optional)

In [ ]:
# Uncomment to run — evaluates the full val split (~10-30 min on one GPU)

# correct_all = 0
# for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc='Val sweep'):
#     choices  = ast.literal_eval(row['choices'])
#     lecture  = str(row['lecture']) if pd.notna(row['lecture']) else ''
#     hint     = str(row['hint'])    if pd.notna(row['hint'])    else ''
#     expected = LETTERS[int(row['answer'])]
#     pred, _, _ = predict_two_stage(
#         image_path=os.path.join(IMAGE_BASE, row['image_path']),
#         question=row['question'],
#         choices=choices,
#         lecture=lecture,
#         hint=hint,
#     )
#     correct_all += int(pred == expected)
# print(f'Full val accuracy: {correct_all}/{len(val_df)} = {correct_all/len(val_df):.2%}')

---
## 13. Generate Submission

Runs two-stage inference over `test.csv` and writes `submission.csv`.
The `answer` column is a **0-based integer index** (matching `sample_submission.csv`), not a letter.

In [ ]:
rows_out = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Test inference'):
    choices = ast.literal_eval(row['choices'])
    lecture = str(row['lecture']) if pd.notna(row['lecture']) else ''
    hint    = str(row['hint'])    if pd.notna(row['hint'])    else ''

    pred_letter, _, _ = predict_two_stage(
        image_path=os.path.join(IMAGE_BASE, row['image_path']),
        question=row['question'],
        choices=choices,
        lecture=lecture,
        hint=hint,
    )
    # Convert letter to 0-based index; fall back to 0 if invalid
    pred_idx = LETTERS.index(pred_letter) if pred_letter in LETTERS else 0
    rows_out.append({'id': row['id'], 'answer': pred_idx})

submission = pd.DataFrame(rows_out)
submission.to_csv('submission.csv', index=False)

print(f'Saved submission.csv  ({len(submission)} rows)')
print('Answer distribution:')
print(submission['answer'].value_counts().sort_index())
submission.head()

---
## Troubleshooting

| Problem | Fix |
|---|---|
| OOM | Set `USE_4BIT = True`; lower `PER_DEVICE_BATCH` to 1; raise `GRAD_ACCUM` to 16 |
| Images not found | Confirm `IMAGE_BASE` is the parent of `images/train/`, `images/val/`, `images/test/` |
| Stage 1 loss stuck | Lower `LEARNING_RATE` to `1e-5`; rationale targets are long |
| Stage 2 predicts same letter always | Try `LORA_R = 32`; check chat template output is correct |
| Slow data loading | Raise `dataloader_num_workers` to 4 |
| Quick iteration | Set `NUM_EPOCHS = 3` first, then scale up |
| Merge LoRA for faster inference | `merged = model.merge_and_unload()` then save as full model |